# 03 Mula sa magulong line list tungo sa malinis na datos — notebook

Na-export na ni Jun ang mga notification pagkatapos ng bagyo mula sa City Health Office:
`santa_luz_linelist_raw.csv`, 637 row na tinype ng maraming kamay. Sa notebook na ito,
gagawin mo itong malinis at de-identified na line list na ginagamit ng bawat susunod na
kabanata — isang maliit at chine-check na hakbang sa bawat pagkakataon.

> **Imbento ang lahat ng narito.** Synthetic na training data ang Santa Luz City, ang
> Bagyong Amihan at ang bawat pangalan, numero ng telepono at address sa hilaw na file.
> *Mukha* lang totoo ang mga pangalan at numero para makapagpraktis kang protektahan ang
> mga ito.

Ang plano: **i-load at tingnan → protektahan ang pagkakakilanlan → ayusin ang petsa →
ayusin ang lugar → ayusin ang mga code → ayusin ang edad → alisin ang duplicates →
i-validate → patunayan.**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Hakbang 1 · I-load at tingnan

Nagsisimula ang file sa isang linyang tala, kaya lalaktawan natin ang isang row.
Sasabihan din natin ang pandas na basahin ang **bawat column bilang text** (`dtype=str`)
at hayaang empty string ang mga blangkong cell (`keep_default_na=False`). Bakit? Kapag
nanghula ang pandas ng type sa magulong file, tahimik itong nagbabago ng mga bagay —
nawawala ang leading zero ng mga numerong `0900...`, nagiging missing value ang `N/A`,
nagiging numero ang `46227`. Gusto nating makita ang gulo nang eksakto kung paano ito
nakita ni Jun, saka tayo mismo ang magpapasya sa bawat pagbabago.

Iilang column lang na hindi nakapagpapakilala ang titingnan natin.

In [ ]:
import hashlib
import hmac
import io
import secrets

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_PATH = "data/synthetic/ph/santa_luz_linelist_raw.csv"

# nilalaktawan ng skiprows=1 ang linyang tala; text ang bawat cell dahil sa dtype=str;
# "" ang blangkong cell imbes na NaN dahil sa keep_default_na=False
raw = pd.read_csv(RAW_PATH, skiprows=1, dtype=str, keep_default_na=False)

print(f"{raw.shape[0]} rows, {raw.shape[1]} columns")
print(list(raw.columns))
raw[["Date Reported", "Sex", "Age", "Barangay", "Disease", "Date of Onset"]].head(8)

## Hakbang 2 · Privacy muna: protektahan ang pagkakakilanlan, saka maglinis

Itinuro ng kabanata 02 ang patakaran: **aalis sa table ang mga identifier bago magsimula
ang analysis.** Kailangan pa natin ang pangalan sa isang trabaho — ang makita ang parehong
pasyenteng dalawang beses na-notify — kaya ginagawa muna natin itong **salted
pseudonymous ID** mula sa kabanata 02: laging pareho ang ID ng parehong inayos na
pangalan, pero kung walang sikretong salt, walang makauulit nito o makababasa ng pangalan
pabalik. Saka natin aalisin ang pangalan, numero ng telepono, address, reporting unit at
free-text remarks (puwedeng magpangalan ng *ibang* tao ang remarks).

Sa parehong cell, ginagawa nating pare-pareho ang pangalan ng mga column bilang
`lower_case_with_underscores`, para madaling i-type.

> **Tanungin si Tala:** "May pandas DataFrame akong `raw` na may text columns na
> `Patient Name`, `Contact No.`, `Address`, `Reporting Unit`, `Remarks` at iba pa.
> Gawa-gawang halimbawa: `Patient Name = 'Juana X. Halimbawa'`. Gamit ang function kong
> `pseudo_id(name, salt)`, magdagdag ng `pseudo_id` column, alisin ang limang column na
> iyon, at gawing snake_case ang lahat ng pangalan ng column."

In [ ]:
SALT = secrets.token_hex(16)  # demo lang: sa totoong trabaho, gumawa nang isang beses at ikandado (kabanata 02)


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # i-normalise ang pangalan
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


df = raw.copy()
df.insert(0, "pseudo_id", df["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
df = df.drop(columns=["Patient Name", "Contact No.", "Address", "Reporting Unit", "Remarks"])

# "Date of Onset" -> "date_of_onset", "Did Cleanup Work" -> "did_cleanup_work"
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
print(list(df.columns))

# I-check: parehong pangalan na na-type sa dalawang paraan -> parehong ID; ibang pangalan -> ibang ID; wala nang identifier
assert pseudo_id("Juana X. Halimbawa", SALT) == pseudo_id("JUANA X  HALIMBAWA", SALT)  # gawa-gawang pangalan
assert pseudo_id("Juana X. Halimbawa", SALT) != pseudo_id("Juan X. Halimbawa", SALT)
for column in ["patient_name", "contact_no", "address", "reporting_unit", "remarks"]:
    assert column not in df.columns
print("Naalis na ang mga identifier; ganito ang itsura ng ID:", df["pseudo_id"].iloc[0])

## Hakbang 3 · Nagkamali si Tala: ang tahimik na pagbaligtad ng petsa

Iba't iba ang style ng mga petsa ni Jun: `07/24/2026`, `7/24/2026`, `2026-07-24`,
`24-Jul-26`, `July 24, 2026`, at mga numerong gaya ng `46227`.

> **Tanungin si Tala:** "May mga petsa sa iba't ibang format ang column kong
> `date_of_onset`. I-convert ito sa datetime."

Mabilis sumagot si Tala: *"Gamitin ang `pd.to_datetime(..., format='mixed', dayfirst=True)`
— araw muna ang isinusulat ng karamihan sa mundo."* Tumatakbo ang code nang walang error.
Iyan ang panganib. Inilalagay natin ang sagot ni Tala sa hiwalay na variables para hindi
nito masira ang datos natin, at tine-test natin ito gamit ang dalawang tanong ng isang
nurse: **ilang petsa ang pumalya?** at **may nagkasakit ba pagkatapos magpakonsulta?**

In [ ]:
# Unang sagot ni Tala -- nakahiwalay para hindi masira ang df
naive_onset = pd.to_datetime(df["date_of_onset"], format="mixed", dayfirst=True, errors="coerce")
naive_consult = pd.to_datetime(df["date_consulted"], format="mixed", dayfirst=True, errors="coerce")

print("Mga petsa ng onset na naging NaT (not a time):", naive_onset.isna().sum())
print("Rows na ang onset ay PAGKATAPOS ng konsulta:", (naive_onset > naive_consult).sum())

swapped = naive_onset > naive_consult
pd.DataFrame({"onset (na-type)": df.loc[swapped, "date_of_onset"],
              "konsulta (na-type)": df.loc[swapped, "date_consulted"],
              "Basa ni Tala sa konsulta": naive_consult[swapped].dt.date}).head(6)

Dalawang problema, parehong tahimik:

1. **22 petsa ng onset ang naging `NaT`** — ang mga Excel serial number (`46227`). Para sa
   pandas, hindi talaga petsa ang mga ito.
2. **56 na pasyente ang "nagkasakit pagkatapos nilang magpakonsulta".** Dahil sa
   `dayfirst=True`, binasa bilang 8 Enero ang `8/1/2026` (1 Agosto). Tama pa ring nabasa
   ang mga petsang gaya ng `7/29/2026` (walang buwang 29), kaya nagtatago ang mali sa
   gitna ng maayos na rows. 2 lang sa 56 ang totoong problema sa file ni Jun; kay Tala
   ang 54.

Sa Pilipinas, buwan/araw/taon ang style kapag may slash, at kinumpirma ni Jun na ganito
ang lahat ng slash date sa export na ito. Kaya magsusulat tayo ng **isang patakaran bawat
format** at hindi tayo manghuhula.

> **Tanungin si Tala (ikalawang subok):** "Sumulat ng function na `parse_dates(s)` para sa
> pandas Series ng mga petsang text. Mga format: `M/D/YYYY` (buwan muna, opsiyonal ang
> leading zero), `YYYY-MM-DD`, `DD-Mon-YY` (20YY ang mga taon), `Month D YYYY` may comma
> man o wala, at 5-digit na Excel serial number (bilang ng araw mula 1899-12-30). Ang
> blangko ay missing. I-parse nang tahasan ang bawat format; huwag hayaang manghula ang
> pandas."

In [ ]:
def parse_dates(s: pd.Series) -> pd.Series:
    """I-parse ang mga style ng petsa ni Jun, isang tahasang format sa bawat pagkakataon."""
    s = s.str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    serial = s.str.fullmatch(r"\d{5}")  # binibilang ng Excel ang araw mula 30 Dis 1899
    out[serial] = pd.to_datetime(s[serial].astype(int), unit="D", origin="1899-12-30")
    for fmt in ["%m/%d/%Y", "%Y-%m-%d", "%d-%b-%y", "%B %d %Y"]:
        attempt = pd.to_datetime(s.str.replace(",", "", regex=False), format=fmt, errors="coerce")
        out = out.fillna(attempt)  # pinupunan lang ang mga petsang kulang pa
    return out


DATE_COLUMNS = {"date_reported": "date_report", "date_of_onset": "date_onset",
                "date_consulted": "date_consult", "date_lab_confirmed": "date_lab_confirmed"}
for old, new in DATE_COLUMNS.items():
    text = df[old]
    parsed = parse_dates(text)
    not_understood = parsed.isna() & text.str.strip().ne("")
    assert not_understood.sum() == 0, text[not_understood].unique()
    df[new] = parsed
    print(f"{new:<20} blangko: {text.str.strip().eq('').sum():>3}   na-parse: {parsed.notna().sum()}")

**I-check natin.** Una, maliit na kasong kayang i-check sa tingin: ang parehong araw na isinulat
sa pitong paraan. Saka ang dalawang tanong kanina.

In [ ]:
same_day = pd.Series(["46227", "7/24/2026", "07/24/2026", "2026-07-24",
                      "24-Jul-26", "July 24 2026", "July 24, 2026", ""])
result = parse_dates(same_day)
print(result.dt.date.tolist())
assert (result[:7] == pd.Timestamp("2026-07-24")).all()
assert pd.isna(result[7])

print("Mga petsa ng onset na NaT:", df["date_onset"].isna().sum())
print("Rows na ang onset ay PAGKATAPOS ng konsulta:", (df["date_onset"] > df["date_consult"]).sum())

Mula 56 na imposibleng row, **2** na lang. Totoong typo ang dalawang iyon sa hilaw na file
(onset na na-type nang isang buwang huli) — aayusin natin sila sa Hakbang 7.

## Hakbang 4 · Iisang barangay, maraming spelling

`Ilog`, `Brgy. Ilog`, `ILOG`, `Bgy Ilog ` (may space sa dulo), `Barangay Ilog`… pati
`Sto. Nino` para sa Santo Niño at ilang typo. Gumagawa tayo ng **key**: alisin ang accent,
gawing maliit na titik, alisin ang prefix na `barangay/brgy/bgy`, isulat ang `sto` bilang
`santo`, gawing space ang mga gitling at tuldok, at ayusin ang tatlong kilalang typo.
Ginagawa rin natin ang parehong key mula sa opisyal na listahan ng barangay at doon tayo
nagja-join, na nagdadala rin ng `barangay_id` at `band`.

**I-check natin:** titigil ang notebook sa isang `assert` kapag may row na walang nakitang
barangay. Bantayan din ang bilang: nagiging 23 barangay ang mga spelling, dahil ang
ika-24, ang Bukid sa kabundukan, ay walang kahit isang notification. Tandaan iyan sa
kabanata 04 — kasama pa rin sa table ang barangay na zero ang kaso.

In [ ]:
TYPOS = {"pobalcion": "poblacion", "dalampasig": "dalampasigan", "kawayan": "kawayanan"}


def barangay_key(s: pd.Series) -> pd.Series:
    """Key na hindi naaapektuhan ng spelling: parehong nagiging 'santo nino' ang 'Brgy. Sto. Niño ' at 'santo nino'."""
    key = (s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")  # ñ -> n
           .str.lower().str.strip()
           .str.replace(r"^(barangay|brgy\.?|bgy\.?)\s+", "", regex=True)
           .str.replace(r"^sto\.?\s+", "santo ", regex=True)
           .str.replace(r"[-.]", " ", regex=True)
           .str.split().str.join(" "))
    return key.replace(TYPOS)


barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")
barangays["key"] = barangay_key(barangays["barangay"])

df = df.rename(columns={"barangay": "barangay_raw"})
df["key"] = barangay_key(df["barangay_raw"])
df = df.merge(barangays[["key", "barangay_id", "barangay", "band"]], on="key", how="left")

unmatched = df.loc[df["barangay_id"].isna(), "barangay_raw"].unique()
assert len(unmatched) == 0, unmatched  # I-check: nakita ng bawat row ang barangay nito
print("Hilaw na spelling:", df["barangay_raw"].nunique(), "-> opisyal na barangay:", df["barangay"].nunique())

# Ilan ang spelling ng bawat barangay? Chart na baka gustong ipakita ni Jun sa mga encoder.
spellings = df.groupby("barangay")["barangay_raw"].nunique().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(spellings.index, spellings.values, color="#0038A8")
ax.set_xlabel("Iba't ibang spelling sa export ni Jun")
ax.set_title("Iisang barangay, maraming spelling")
plt.tight_layout()
plt.show()

print(df.loc[df["barangay"] == "Santo Niño", "barangay_raw"].unique())

## Hakbang 5 · Mga code sa Taglish: sex, sakit, classification, oo/hindi, outcome, severity

Bawat field ay may maliit na **lookup table** (isang dictionary) o patakaran. Nagiging
missing ang anumang wala sa table, kaya mahuhuli ng mga check sa dulo ng cell ang anumang
spelling na nakalimutan natin.

| Field | Halimbawang hilaw na value | Malinis |
|---|---|---|
| Sex | `M`, `male`, `lalaki` / `F`, `Female`, `babae` | `M` / `F` |
| Sakit | `Lepto`, `LEPTO`… / `DENGUE`, `Dengue Fever`, `DF` | `leptospirosis` / `dengue` |
| Classification | `lab confirmed`, `Conf` / `prob.` / `sus`, `Suspected` | `confirmed` / `probable` / `suspect` |
| Oo/hindi | `Y`, `yes`, `oo` / `N`, `no`, `hindi` / blangko, `N/A` | `1` / `0` / missing |
| Outcome | `buhay`, `A` / `Namatay`, `Expired`, `D` | `alive` / `died` |

In [ ]:
sex = df["sex"].str.strip().str.lower()
df["sex"] = sex.map({"m": "M", "male": "M", "lalaki": "M", "f": "F", "female": "F", "babae": "F"})

disease = df["disease"].str.strip().str.lower()
df["disease"] = np.select([disease.str.startswith("lep"),
                           disease.str.startswith("deng") | disease.eq("df")],
                          ["leptospirosis", "dengue"], default=None)

cls = df["classification"].str.strip().str.lower()
df["case_class"] = np.select([cls.str.contains("conf"), cls.str.startswith("prob"),
                              cls.str.startswith("sus")],
                             ["confirmed", "probable", "suspect"], default=None)

YES_NO = {"y": 1, "yes": 1, "oo": 1, "n": 0, "no": 0, "hindi": 0}  # blangko at "n/a" -> missing
for old, new in {"admitted": "hospitalised", "waded_in_floodwater": "waded_floodwater",
                 "did_cleanup_work": "cleanup_work"}.items():
    df[new] = df[old].str.strip().str.lower().map(YES_NO).astype("Int64")

outcome = df["outcome"].str.strip().str.lower()
df["outcome"] = outcome.map({"alive": "alive", "a": "alive", "buhay": "alive",
                             "died": "died", "d": "died", "namatay": "died", "expired": "died"})

sev = df["dengue_severity"].str.strip().str.lower()
df["dengue_severity"] = np.select(  # mahalaga ang pagkakasunod: may "with" din sa loob ng "without"
    [sev.str.contains("severe") | sev.eq("sd"),
     sev.str.contains("without") | sev.str.contains("w/o", regex=False) | sev.eq("no ws"),
     sev.str.contains("with") | sev.str.contains("w/", regex=False) | sev.eq("ws+")],
    ["severe", "no_warning", "warning_signs"], default=None)
df.loc[df["disease"] != "dengue", "dengue_severity"] = None

# Check: walang nakalusot sa mga lookup table
for column in ["sex", "disease", "case_class", "outcome", "hospitalised"]:
    assert df[column].notna().all(), column
assert df.loc[df["disease"] == "dengue", "dengue_severity"].notna().all()
print(df["disease"].value_counts().to_dict(), df["sex"].value_counts().to_dict())

## Hakbang 6 · Edad: "34 y/o", "8 mos" at isang 150

Kunin ang unang buong numero. Kung may `mo` sa text, buwan iyon, kaya hatiin sa 12 at
itira ang buong taon (`8 mos` → 0, gaya sa case investigation form para sa sanggol).
Imposible ang edad na lampas 110, kaya nagiging missing ito — sa totoong buhay, tatawagan
mo ang reporting unit. Saka natin ilalagay ang mga edad sa karaniwang mga grupo.

In [ ]:
def parse_age(s: pd.Series) -> pd.Series:
    """Buong taon mula sa text na gaya ng '34', '34 y/o', '34yo', '8 mos', '8 months'."""
    text = s.str.strip().str.lower()
    number = text.str.extract(r"^(\d+)")[0].astype("Int64")
    years = number.where(~text.str.contains("mo"), number // 12)  # buwan -> taon
    return years.where(years <= 110).astype("Int64")             # 150 -> missing


# Check sa kasong kaya mong gawin sa isip
print(parse_age(pd.Series(["34", "34 y/o", "34yo", "8 mos", "18 months", "150", ""])).tolist())

df["age"] = parse_age(df["age"])
df["age_group"] = pd.cut(df["age"], bins=[-1, 4, 14, 24, 44, 64, 200],
                         labels=["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]).astype(object)
print("Walang edad:", df["age"].isna().sum(), "| pinakamatanda:", df["age"].max())

## Hakbang 7 · Duplicates, imposibleng petsa, at bagong ID

Madali ang **eksaktong kopya**: `df.duplicated()`. Mas mahirap ang **halos-duplicate**:
nag-re-notify ang City Health Office ng ilang pasyente na muling na-type ang pangalan,
ibang style ng petsa at ibang spelling ng barangay. Pagkatapos ng paglilinis natin,
*mukhang* pareho na ang mga row na iyon sa mahahalagang column, kaya nagde-de-duplicate
tayo ayon sa **(pseudo ID, sakit, petsa ng onset)** at itinitira ang una.

Saka ang huling tatlong patakaran:

- **Imposibleng petsa:** kapag lampas sa konsulta ang onset (o lampas sa petsa ng report
  kapag blangko ang konsulta), gawing missing ang onset — at, sa totoong buhay, tawagan ang
  reporting unit.
- **Pseudonymous case ID** na `SLC-0001`, `SLC-0002`, … ayon sa pagkakasunod sa file.
- **Mga column** sa napagkasunduang pagkakasunod. Naiiwan ang pseudo ID, ang hilaw na text
  at ang mga identifier.

In [ ]:
print("Eksaktong kopya:", df.duplicated().sum())
duplicate = df.duplicated(subset=["pseudo_id", "disease", "date_onset"], keep="first")
print("Parehong tao + sakit + onset (eksakto at halos):", duplicate.sum())
df = df[~duplicate].reset_index(drop=True)

later = df["date_consult"].fillna(df["date_report"])
impossible = df["date_onset"] > later
print("Onset pagkatapos ng konsulta/report -> ginawang missing:", impossible.sum())
df.loc[impossible, "date_onset"] = pd.NaT

df["case_id"] = [f"SLC-{i:04d}" for i in range(1, len(df) + 1)]

COLUMNS = ["case_id", "disease", "case_class", "date_onset", "date_consult", "date_report",
           "date_lab_confirmed", "age", "age_group", "sex", "barangay_id", "barangay", "band",
           "hospitalised", "outcome", "dengue_severity", "waded_floodwater", "cleanup_work"]
clean = df[COLUMNS]
print(clean.shape)
clean.head()

## Hakbang 8 · Mga validation rule bilang assertion

Ang **validation rule** ay pangungusap tungkol sa datos na dapat laging totoo, na
isinulat bilang `assert`. Kapag may lumabag na export sa hinaharap, malakas na titigil ang
notebook imbes na tahimik na gumawa ng maling report. Basahin nang malakas ang bawat linya
— epidemiology ang mga ito, hindi lang code.

In [ ]:
assert clean["case_id"].is_unique
assert clean["date_report"].between(pd.Timestamp("2026-07-20"), pd.Timestamp("2026-09-30")).all()
both = clean.dropna(subset=["date_onset", "date_consult"])
assert (both["date_onset"] <= both["date_consult"]).all()          # nagkasakit bago magpakonsulta
assert (clean["date_consult"] <= clean["date_report"]).where(clean["date_consult"].notna(), True).all()
assert clean["age"].dropna().between(0, 110).all()
assert set(clean["sex"]) == {"M", "F"}
confirmed = clean["case_class"] == "confirmed"
assert clean.loc[confirmed, "date_lab_confirmed"].notna().all()     # confirmed = may lab result
assert clean.loc[clean["disease"] == "dengue", "waded_floodwater"].isna().all()
print("Pumasa ang lahat ng validation rule.")
print(clean["disease"].value_counts().to_dict())
print("Kulang: onset", clean["date_onset"].isna().sum(), "| konsulta", clean["date_consult"].isna().sum(),
      "| edad", clean["age"].isna().sum())

## Hakbang 9 · Patunayan

Tanong ni Doc Mon: *"Pareho ba ang sa iyo sa opisyal na malinis na file?"* Sine-save natin
ang table ayon sa sinasabi ng rule C8 (`to_csv(index=False)`, dito sa memory), binabasa
ulit, at hinahayaan ang pandas na ikumpara ito sa `santa_luz_linelist.csv`, cell por cell.
Walang ipini-print ang `assert_frame_equal` kapag magkapareho ang mga table, at
ipinapaliwanag nito ang unang pagkakaiba kapag hindi.

In [ ]:
csv_text = clean.to_csv(index=False, lineterminator="\n")  # ang ise-save natin
ours = pd.read_csv(io.StringIO(csv_text))
official = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

pd.testing.assert_frame_equal(ours, official)
print(f"Kaparehong-kapareho ng santa_luz_linelist.csv: {ours.shape[0]} rows x {ours.shape[1]} columns")

## Ikaw naman

1. **Mga buwan sa Filipino.** Nag-type ang isang bagong encoder ng `Hulyo 24, 2026`. Sa
   bagong cell, patakbuhin ang `parse_dates(pd.Series(["Hulyo 24, 2026"]))`. Ano ang
   lumabas, at pinoprotektahan ka ba ng `assert` sa Hakbang 3? *Hint: palitan ang `Hulyo`
   ng `July` (at ang `Agosto` ng `August`) gamit ang `.str.replace` bago mag-parse, saka
   patakbuhin ulit ang check na pitong paraan.*
2. **Bagong spelling.** Ano ang ibinabalik ng `barangay_key(pd.Series(["Brgy Tabing  Ilog"]))`?
   Tugma ba ito sa `tabing ilog`? Subukan naman ang `"Tabing-ilog (riverside)"`. *Hint:
   hindi kayang ayusin ng key ang lahat. Hanapin ang row gamit ang
   `.str.contains("Tabing")`, at magdagdag lang ng patakaran kapag alam mo ang ibig sabihin
   ng encoder.*
3. **Bilangin ang trabaho.** Ilang row ang binago ng bawat hakbang? *Hint: ikumpara ang
   `len(raw)` sa `len(clean)`, at bilangin ang `naive_onset.isna()` laban sa
   `df["date_onset"].isna()`. Isulat ang mga numero sa tatlong-linyang tala para kay Jun.*